In [1]:
import requests

def generate_embedding(query: str) -> list[float]:
    """Generate embedding from query."""
    url = "https://ollama-2.prod.dart.prv.vai-compute.com/api/embed"
    params = {"model": "jina/jina-embeddings-v2-base-en", "input": query}
    response = requests.post(url, json=params, timeout=360)

    if response.status_code in [200]:
        embedding: list[float] = response.json()["embeddings"][0]
        return embedding
    else:
        return []

/Users/rorydonovan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
import json
from uuid import uuid4

# name: create
# properties: {"require_arg": {"arg_name": {"type": "string", "description": "The url of the webpage to crawl."}}}

name = "Crawl Webpage"  # Name, shown to the user
tool_name = "crawler_post_parse"  # tool_name, shown to the LLM
tool_description = "A tool to crawl a webpage and extract the text content from it. It can be used to gather information from a specific URL."
tool_description_embedding = generate_embedding(tool_description)
tool_description_for_llm = tool_description  # Description passed to the LLM for it to choose this tool, should be short, about 1 sentence.

# python function to invoke
tool_module_path="vaiflow.tools.http_api"
tool_function_name="post"

# parameters for the tool which do not come from the LLM
fixed_params = {
    "url": "https://crawler.prod.dart.prv.vai-compute.com/crawler/v1/parse",
    "headers": {},
    # body gets merged with tool_args and submitted as json
    "body": {}
}

# llm tool definition, parameters from here are filled in by LLM and given as tool_args to the tool
tool_definition = {
    "type": "function",
    "function": {
        "name": tool_name,
        "description": tool_description_for_llm,
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "The url of the webpage to crawl.",
                }
            },
            "required": ["url"],
            "additionalProperties": False,
        },
    },
}

# this is the final tool node in the graph
tool_node = dict(
    uuid=str(uuid4()),
    name=name,
    tool_name=tool_name,
    tool_description=tool_description,
    tool_description_embedding=tool_description_embedding,
    tool_definition=json.dumps(
        {
            "type": "function",
            "function": {
                "name": tool_name,
                "description": tool_description_for_llm,
                "strict": True,
            },
        }
    ),
    tool_module_path=tool_module_path,
    tool_function_name=tool_function_name,
    tool_fixed_params=json.dumps(fixed_params),
)

In [3]:
print(json.dumps(tool_node, indent=2))

{
  "uuid": "1e951cff-6442-4173-9697-e6846058a7a6",
  "name": "Crawl Webpage",
  "tool_name": "crawler_post_parse",
  "tool_description": "A tool to crawl a webpage and extract the text content from it. It can be used to gather information from a specific URL.",
  "tool_description_embedding": [
    -0.039270006,
    -0.06210437,
    0.0017595981,
    0.0015976299,
    -0.038825955,
    0.014828236,
    0.034244757,
    0.0108295595,
    0.022567248,
    0.012307948,
    -0.04537294,
    -0.02321512,
    -0.023478435,
    0.00975088,
    -0.06772319,
    0.08852821,
    0.0218987,
    -0.023200817,
    0.0124178445,
    0.028533757,
    -0.0050380277,
    0.0023586208,
    -0.052659906,
    -0.0061829477,
    0.05913712,
    0.036171477,
    0.07296395,
    0.044374406,
    0.07012438,
    0.032405745,
    0.0086526545,
    -0.010687928,
    -0.038579296,
    0.023791842,
    -0.062136125,
    -0.036144126,
    -0.02205051,
    0.019806828,
    0.04460195,
    0.048992567,
    -0.00506

{
  "uuid": "1e951cff-6442-4173-9697-e6846058a7a6",
  "name": "Crawl Webpage",
  "tool_name": "crawler_post_parse",
  "tool_description": "A tool to crawl a webpage and extract the text content from it. It can be used to gather information from a specific URL.",
  "tool_description_embedding": [
  ],
  "tool_definition": "{\"type\": \"function\", \"function\": {\"name\": \"crawler_post_parse\", \"description\": \"A tool to crawl a webpage and extract the text content from it. It can be used to gather information from a specific URL.\", \"strict\": true}}",
  "tool_module_path": "vaiflow.tools.http_api",
  "tool_function_name": "post",
  "tool_fixed_params": "{\"url\": \"https://crawler.prod.dart.prv.vai-compute.com/crawler/v1/parse\", \"headers\": {}, \"body\": {}}"
}

In [ ]:
# Insert the tool node in to the graph, merge on tool_name so it updates the original node
import os
import neo4j
import dotenv

dotenv.load_dotenv()

connection = neo4j.GraphDatabase.driver(
    "neo4j+s://neo4j.prod.dart.prv.vai-compute.com:7687",
    auth=(os.environ['VAI_NEO4J_USER'], os.environ['VAI_NEO4J_PASSWORD']), database="vaiflowrory"
)

# Remove uuid from the update props to the UUID on the node doesn't change
create_props = tool_node.copy()
# tool_node.pop("uuid", None)
update_props =  tool_node

with connection.session() as session:
    session.run("""
        MERGE (t:Tool {tool_name: $tool_name})
        ON CREATE SET t += $create_props
        ON MATCH SET t += $update_props
        """,
        {
            "tool_name": tool_node["tool_name"],
            "create_props": create_props,
            "update_props": update_props
        },
    )